# 🛡️ 기능 2. 보안 샌드박스 (Defense Arena) - PDF 격리 업로드 및 파싱 (기초)

본 가이드는 **Paper Agent** 프로젝트의 **보안 격리 샌드박스** 기능 중 **PDF 격리 업로드 및 OS Path Guard 검증, 텍스트 분할 및 pgvector 데이터베이스 3072차원 임베딩 생성**의 핵심 동작 원리를 다룹니다.

학습 집중에 불필요한 예외 처리 및 Mock 폴백 코드를 최소화하여, **실제 격리 구역 파일 업로드 핵심 구현 코드의 로직만 간결하고 명확하게 확인**할 수 있도록 작성되었습니다.

---

## 📌 기초 학습 핵심 포인트
1. **백엔드 환경 설정 로드**
   - `backend/.env` 환경 변수와 패키지 경로를 주입합니다.
2. **OS Path Guard 및 격리 저장**
   - 디렉토리 트래버설(Directory Traversal) 공격을 방지하기 위해 `os.path.realpath`를 사용한 경로 검증을 실습합니다.
3. **PDF 텍스트 추출 (`pypdf`)**
   - `pypdf.PdfReader`를 이용해 업로드된 PDF 바이너리 스트림에서 텍스트를 추출하는 백엔드 핵심 방식을 이해합니다.
4. **텍스트 청킹 (`RecursiveCharacterTextSplitter`)**
   - `chunk_size = 1000` 및 `chunk_overlap = 200` 자 기준으로 긴 논문 본문을 분할하는 원리를 실습합니다.
5. **3072차원 pgvector 임베딩 및 인덱싱**
   - OpenAI `text-embedding-3-large` 모델을 기반으로 임베딩을 구성하고 pgvector HNSW 인덱스 조건에 맞게 데이터베이스에 적재하는 논리 구조를 배웁니다.

## 1단계. 백엔드 환경 설정 및 모듈 로드

노트북 실행 위치를 파악하고 백엔드 패키지 경로를 주입한 뒤, 필수 환경변수들을 로드합니다.

In [6]:
import os
import sys
import warnings
from pathlib import Path
from dotenv import load_dotenv

# Pydantic 및 LangChain 연동 UserWarning 무시
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

# 현재 notebooks/defense_arena/ 폴더 경로 기준 백엔드 디렉토리 탐색
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

# 백엔드 .env 로드
env_path = backend_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ 백엔드 환경 설정 로드 완료: {env_path}")

openai_key = os.getenv("OPENAI_API_KEY", "")
database_url = os.getenv("DATABASE_URL", "")

print(f"🔑 OpenAI API Key 설정 여부: {'설정됨(Yes)' if openai_key else '설정안됨(No)'}")
print(f"🗄️ Database URL: {database_url[:35]}..." if database_url else "🗄️ Database URL: 설정안됨")

✅ 백엔드 환경 설정 로드 완료: /Users/pileuszu/Repos/bist-mini-2/backend/.env
🔑 OpenAI API Key 설정 여부: 설정됨(Yes)
🗄️ Database URL: postgresql+asyncpg://postgres:postg...


## 2단계. OS Path Guard 및 격리 저장 실습

사용자가 업로드한 파일명의 상위 경로 접근 공격(Directory Traversal, 예: `../../etc/passwd`)을 차단하기 위한 OS Path Guard 설계 메커니즘을 테스트합니다.
`os.path.realpath`를 사용해 실제 주소의 접두어가 허용된 임시 업로드 디렉토리 영역 내에 있는지 검증합니다.

In [7]:
import os

# 실제 업로드 허용 구역 지정
UPLOAD_DIR = os.path.realpath("./uploads")

def validate_isolated_path(filename: str, session_id: str) -> str:
    """OS Path Guard를 모방한 안전한 격리 경로 검증기입니다.
    """
    session_dir = os.path.join(UPLOAD_DIR, session_id)
    target_path = os.path.realpath(os.path.join(session_dir, filename))
    
    # target_path가 허용된 UPLOAD_DIR로 시작하는지 철저히 체크
    if not target_path.startswith(UPLOAD_DIR):
        raise PermissionError("🚨 OS Path Guard 위반 감지: 허용되지 않는 디렉토리 경로 접근 시도입니다.")
    
    return target_path

# --- 정상 업로드 시나리오 ---
try:
    safe_path = validate_isolated_path("draft_paper.pdf", "session-uuid-1234")
    print(f"✅ 정상 경로 검증 성공: {safe_path}")
except Exception as e:
    print(f"❌ 실패: {e}")

# --- Traversal 공격 시나리오 ---
try:
    attack_filename = "../../dangerous_system_file.conf"
    unsafe_path = validate_isolated_path(attack_filename, "session-uuid-1234")
    print(f"❌ 필터링 실패 (보안 취약): {unsafe_path}")
except PermissionError as e:
    print(f"✅ 보안 가드 작동 성공! 오류 메시지: {e}")

✅ 정상 경로 검증 성공: /Users/pileuszu/Repos/bist-mini-2/notebooks/defense_arena/uploads/session-uuid-1234/draft_paper.pdf
✅ 보안 가드 작동 성공! 오류 메시지: 🚨 OS Path Guard 위반 감지: 허용되지 않는 디렉토리 경로 접근 시도입니다.


## 3단계. pypdf를 활용한 PDF 텍스트 추출 실습

격리 폴더에 적재된 논문 PDF 바이너리 스트림에서 텍스트 콘텐츠를 일련의 문자열로 안전하게 추출합니다.

In [8]:
import io
from pypdf import PdfReader

def extract_text_from_pdf(pdf_bytes: bytes) -> str:
    """PDF 파일 바이트로부터 텍스트 전체를 파싱합니다."""
    reader = PdfReader(io.BytesIO(pdf_bytes))
    extracted_pages = []
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        extracted_pages.append(text)
    return "\n".join(extracted_pages)

# Mock PDF 바이트 또는 실제 PDF 읽기 테스트
# (이해를 위해 텍스트 계열 데이터로 대체 실습하거나, astro.pdf 파일이 있다면 직접 테스트해 볼 수 있습니다)
demo_pdf_path = Path("../gem_service/astro.pdf")
if demo_pdf_path.exists():
    pdf_data = demo_pdf_path.read_bytes()
    text_content = extract_text_from_pdf(pdf_data)
    print(f"✅ PDF 파싱 완료! 추출된 텍스트 크기: {len(text_content)} 자")
    print(f"--- 샘플 텍스트 (앞 200자) ---\n{text_content[:200]}")
else:
    print("⚠️ astro.pdf 파일이 없어 실제 파싱 생략 (대체 텍스트 활용)")
    text_content = "This is a mock academic paper abstract. \nWe present a new learning method that improves SOTA. \nThe methodology details are described in Section 3."
    print(f"--- 샘플 텍스트 (앞 200자) ---\n{text_content[:200]}")

✅ PDF 파싱 완료! 추출된 텍스트 크기: 7237 자
--- 샘플 텍스트 (앞 200자) ---
A S T R O N O M Y
천문학 입문
태양계에서 우주까지
밤하늘의 별 하나에서 시작해 우리은하, 그리고 138억 년 우주
의 역사까지 —
처음 천문학을 만나는 이를 위한 한 권의 안내서.
학습용 개론서 · 전 10장 구성
Introduction to Astronomy
목차C O N T E N T S
이 책의 구성
01 천문학이란 무엇인가
과학으로서의


## 4단계. RecursiveCharacterTextSplitter 기반 청킹 실습

백엔드 [services.py](file:///Users/pileuszu/Repos/bist-mini-2/backend/api/v1/defense_arena/services.py#L188-L191)의 구현 스펙에 맞춰 `RecursiveCharacterTextSplitter`를 이용해 문단을 청킹합니다.
설정 값: `chunk_size = 1000`, `chunk_overlap = 200` 자.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1000글자 크기, 200글자 중첩
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_text(text_content)

print(f"📋 분할된 총 청크 수: {len(chunks)} 개")
if chunks:
    print(f"\n--- [첫 번째 청크] ({len(chunks[0])}자) ---\n{chunks[0]}")
    if len(chunks) > 1:
        print(f"\n--- [두 번째 청크] ({len(chunks[1])}자) ---\n{chunks[1]}")

📋 분할된 총 청크 수: 9 개

--- [첫 번째 청크] (966자) ---
A S T R O N O M Y
천문학 입문
태양계에서 우주까지
밤하늘의 별 하나에서 시작해 우리은하, 그리고 138억 년 우주
의 역사까지 —
처음 천문학을 만나는 이를 위한 한 권의 안내서.
학습용 개론서 · 전 10장 구성
Introduction to Astronomy
목차C O N T E N T S
이 책의 구성
01 천문학이란 무엇인가
과학으로서의 천문학, 거리의 단위와 빛
p.3
02 태양계의 구조
태양을 중심으로 한 가족, 행성의 분류
p.4
03 태양 — 우리의 별
핵융합, 내부 구조, 태양 활동
p.5
04 지구형 행성
수성·금성·지구·화성의 세계
p.6
05 목성형 행성과 작은 천체들
거대 행성, 위성·소행성·혜성
p.7
06 별의 탄생과 죽음
성운에서 백색왜성·블랙홀까지
p.8
07 은하와 우주의 거대 구조
우리은하, 은하군과 은하단
p.9
08 우주의 기원과 미래
빅뱅, 암흑물질·암흑에너지, 관측의 도구
p.10
"우리는 별의 먼지로 이루어져 있다."
우리 몸을 이루는 원소 대부분은 오래전 별의 내부에서 만들어졌습니다. 천문학을 배운다는 것은
곧 우리 자신의 기원을 더듬어 가는 일이기도 합니다. 이 책은 가까운 태양계에서 출발해 점점 시
야를 넓혀, 마지막에는 우주 전체의 탄생과 미래까지 함께 살펴봅니다.
2 천문학 입문
01C H A P T E R  0 1
천문학이란 무엇인가
천문학(天文學, Astronomy)은 우주에 존재하는 천체와 그곳에서 일어나는 현상을 연구하는 자연과학입
니다. 별과 행성, 은하, 그리고 우주 전체의 구조와 역사까지가 모두 그 대상입니다.
가장 오래된 과학
천문학은 인류가 가장 오래전부터 연구해 온 학문 중 하나입니다. 고대 문명은 별과 태양, 달의 움직임을 관찰
해 달력을 만들고 농사 시기를 정했으며, 밤하늘의 별자리를 이용해 사막과 바다에서 길을 찾았습니다. 즉 천
문학은 처음부터 실용적인 필요와 함께 발전해 온 셈입니다.
관측과 

## 5단계. OpenAI text-embedding-3-large 3072차원 임베딩 생성

추출된 각 청크를 OpenAI의 대형 임베딩 모델 `text-embedding-3-large`을 통해 3072차원 벡터로 변환하는 메커니즘을 학습합니다.
실제 pgvector 테이블 구축 시 코사인 유사도 검색 최적화를 위해 **HNSW 인덱싱** 방식을 활용하며, 생성 매개변수로 `WITH (m = 16, ef_construction = 64)` 설정을 필수로 적용해야 합니다.

In [10]:
from langchain_openai import OpenAIEmbeddings

if not openai_key:
    print("🚨 OPENAI_API_KEY가 없습니다. 실습을 위해 가상 벡터로 대체합니다.")
    # Mock 임베딩
    mock_vector = [0.01] * 3072
    print(f"✅ 생성된 임베딩 차원: {len(mock_vector)} 차원 (Mock)")
else:
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-large",
        api_key=openai_key
    )
    # 첫 청크 임베딩 수행
    sample_text = chunks[0] if chunks else "Hello paper agent"
    vector = embeddings.embed_query(sample_text)
    print(f"✅ 생성된 임베딩 차원: {len(vector)} 차원")
    print(f"📊 벡터 샘플 (앞 5개 요소): {vector[:5]}")
    
print("\n💡 [pgvector 데이터베이스 적재 표준 SQL 스펙]")
print("""CREATE TABLE defense_arena_chunk (
    id SERIAL PRIMARY KEY,
    session_id UUID NOT NULL REFERENCES defense_arena_session(session_id) ON DELETE CASCADE,
    chunk_index INT NOT NULL,
    text_chunk TEXT NOT NULL,
    embedding vector(3072) NOT NULL
);

-- 코사인 유사도와 속도를 동시에 충족하는 HNSW 인덱스 생성
CREATE INDEX ON defense_arena_chunk 
USING hnsw (embedding vector_cosine_ops) 
WITH (m = 16, ef_construction = 64);
""")

✅ 생성된 임베딩 차원: 3072 차원
📊 벡터 샘플 (앞 5개 요소): [0.0262451171875, -0.0491943359375, -0.0100555419921875, -0.0193939208984375, -0.0180511474609375]

💡 [pgvector 데이터베이스 적재 표준 SQL 스펙]
CREATE TABLE defense_arena_chunk (
    id SERIAL PRIMARY KEY,
    session_id UUID NOT NULL REFERENCES defense_arena_session(session_id) ON DELETE CASCADE,
    chunk_index INT NOT NULL,
    text_chunk TEXT NOT NULL,
    embedding vector(3072) NOT NULL
);

-- 코사인 유사도와 속도를 동시에 충족하는 HNSW 인덱스 생성
CREATE INDEX ON defense_arena_chunk 
USING hnsw (embedding vector_cosine_ops) 
WITH (m = 16, ef_construction = 64);

